# Ketamine Docking Pipeline - Google Colab

Bu notebook, ketamin moleküler docking pipeline'ını Google Colab'da çalıştırmanızı sağlar.

**⚠️ Önemli Notlar:**
- Runtime: ~1-2 saat (tüm hedefler için)
- Colab ücretsiz sürümünde çalışır
- Sonuçları indirmeyi unutmayın (Colab geçici)
- GPU gerekmez (CPU yeterli)

**Adımlar:**
1. Setup: Tüm bağımlılıkları yükle
2. Download: Pipeline kodlarını indir
3. Run: Pipeline'ı çalıştır
4. Download Results: Sonuçları bilgisayarına indir

## 1️⃣ Setup - Bağımlılıkları Yükle

Bu adım ~5-10 dakika sürer.

In [ ]:
# Önce çalışma dizinini kontrol et
!pwd
!ls -la

In [ ]:
# Python paketlerini yükle
print("📦 Python paketleri yükleniyor...")

!pip install -q biopython rdkit pandas pyyaml openpyxl matplotlib seaborn scipy numpy

print("✓ Python paketleri yüklendi")

In [ ]:
# AutoDock Vina'yı yükle (binary download)
print("🔬 AutoDock Vina yükleniyor...")

import os
import urllib.request

# Vina binary indir
vina_url = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"

if not os.path.exists("/usr/local/bin/vina"):
    print("  Downloading Vina...")
    urllib.request.urlretrieve(vina_url, "/tmp/vina")
    !chmod +x /tmp/vina
    !sudo mv /tmp/vina /usr/local/bin/vina
    print("  ✓ Vina installed")
else:
    print("  ✓ Vina already installed")

# Test
!vina --version
print("✓ AutoDock Vina yüklendi")

In [ ]:
# Open Babel yükle (PDBQT dönüşüm için)
print("🧪 Open Babel yükleniyor...")

!apt-get update -qq
!apt-get install -qq -y openbabel

# Test
!obabel --version
print("✓ Open Babel yüklendi")

In [ ]:
# Kurulumu doğrula
print("\n" + "="*70)
print("KURULUM KONTROLÜ")
print("="*70)

import sys

# Python version
print(f"Python: {sys.version.split()[0]}")

# Paketler
packages = ['Bio', 'rdkit', 'pandas', 'yaml', 'numpy', 'scipy', 'matplotlib']
for pkg in packages:
    try:
        __import__(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg} EKSIK!")

# External tools
!which vina > /dev/null && echo "✓ AutoDock Vina" || echo "✗ Vina EKSIK!"
!which obabel > /dev/null && echo "✓ Open Babel" || echo "✗ Open Babel EKSIK!"

print("="*70)
print("✓ Kurulum tamamlandı!")
print("="*70)

## 2️⃣ Pipeline Kodlarını İndir

GitHub'dan pipeline kodlarını çek.

In [ ]:
# GitHub reposunu clone et
!rm -rf comp-chem-scripts  # Önceki varsa sil

print("📥 Repository indiriliyor...")
!git clone -b claude/ketamine-docking-pipeline-011CV5ZKC6fFJBRhdCG2Mtt6 https://github.com/egundeger/comp-chem-scripts.git

# Ketamine docking dizinine geç
%cd comp-chem-scripts/ketamine-docking

print("\n✓ Kodlar indirildi!")
print("\nKlasör içeriği:")
!ls -la

## 3️⃣ Pipeline'ı Çalıştır

### Seçenek A: Tüm Pipeline (Tavsiye Edilen)

Tüm adımları otomatik çalıştırır (~1-2 saat)

In [ ]:
# Tüm pipeline'ı çalıştır (non-interactive Colab versiyonu)
# Not: Bu adım 1-2 saat sürebilir

print("🚀 Pipeline başlatılıyor...\n")
print("⏱️  Tahmini süre: 1-2 saat")
print("⚠️  Bağlantınızı açık tutun!\n")

# Colab versiyonunu kullan (otomatik, kullanıcı girişi gerektirmeyen)
!python run_pipeline_colab.py

### Seçenek B: Adım Adım Çalıştırma

Her adımı ayrı ayrı çalıştırıp kontrol edebilirsiniz.

In [ ]:
# Adım 1: PDB yapılarını indir
print("⬇️ Adım 1: PDB yapıları indiriliyor...\n")
!python scripts/1_download_structures.py

In [ ]:
# Adım 2: Ketamin ligandını hazırla
print("🧪 Adım 2: Ketamin ligandı hazırlanıyor...\n")
!python scripts/2_prepare_ligand.py

In [ ]:
# Adım 3: Proteinleri hazırla
print("🔧 Adım 3: Proteinler hazırlanıyor...\n")
!python scripts/3_prepare_proteins.py

In [ ]:
# Adım 4: Docking yap (EN UZUN ADIM - 1+ saat)
print("🎯 Adım 4: Docking başlatılıyor...\n")
print("⚠️ Bu adım 1+ saat sürebilir\n")
!python scripts/4_run_docking.py

In [ ]:
# Adım 5: Sonuçları analiz et
print("📊 Adım 5: Sonuçlar analiz ediliyor...\n")
!python scripts/5_analyze_results.py

## 4️⃣ Sonuçları Görüntüle

In [ ]:
# Genel özeti göster
import glob

summary_files = glob.glob('data/results/reports/overall_summary_*.txt')
if summary_files:
    latest_summary = sorted(summary_files)[-1]
    print("="*70)
    print("GENEL ÖZET")
    print("="*70)
    with open(latest_summary, 'r') as f:
        print(f.read())
else:
    print("❌ Henüz sonuç bulunamadı. Pipeline'ı çalıştırdınız mı?")

In [ ]:
# Özet tabloyu göster
import pandas as pd
import glob

csv_files = glob.glob('data/results/reports/summary_*.csv')
if csv_files:
    latest_csv = sorted(csv_files)[-1]
    df = pd.read_csv(latest_csv)
    print("\n📊 Docking Sonuçları Özet Tablosu:\n")
    print(df.to_string(index=False))
    
    # Grafik çiz
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Her target için en iyi affinity
    best_per_target = df.groupby('Target')['Best Affinity (kcal/mol)'].min()
    
    colors = ['green' if x <= -7 else 'orange' if x <= -5 else 'red' 
              for x in best_per_target.values]
    
    best_per_target.plot(kind='bar', ax=ax, color=colors)
    ax.set_ylabel('Binding Affinity (kcal/mol)')
    ax.set_xlabel('Target')
    ax.set_title('En İyi Bağlanma Affinitesi (Hedef Bazında)')
    ax.axhline(y=-7, color='green', linestyle='--', label='Güçlü bağlanma')
    ax.axhline(y=-5, color='orange', linestyle='--', label='Zayıf bağlanma')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("❌ CSV sonuç dosyası bulunamadı")

## 5️⃣ Sonuçları İndir

Colab geçici olduğu için sonuçları bilgisayarınıza indirin!

In [ ]:
# Sonuçları ZIP olarak hazırla
import shutil
import os
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f'ketamine_docking_results_{timestamp}'

print(f"📦 Sonuçlar paketleniyor: {zip_name}.zip\n")

# ZIP oluştur
shutil.make_archive(zip_name, 'zip', 'data/results')

print(f"✓ Paketleme tamamlandı: {zip_name}.zip")
print(f"Dosya boyutu: {os.path.getsize(f'{zip_name}.zip') / 1024 / 1024:.2f} MB\n")

# İndir
from google.colab import files

print("⬇️ İndirme başlatılıyor...")
files.download(f'{zip_name}.zip')

print("\n✓ İndirme tamamlandı!")
print("\nZIP içeriği:")
print("  - reports/: Tüm raporlar (TXT, Excel, CSV)")
print("  - nmda/: NMDA docking sonuçları")
print("  - egfr/: EGFR docking sonuçları")
print("  - csnk1d/: CSNK1D docking sonuçları")
print("  - all_results.json: Ham veri")

In [ ]:
# Alternatif: Sadece raporları indir (daha küçük)
import glob
import os
from google.colab import files

print("📄 Sadece raporlar indiriliyor...\n")

report_files = glob.glob('data/results/reports/*')

for report_file in report_files:
    if os.path.isfile(report_file):
        print(f"  Downloading: {os.path.basename(report_file)}")
        files.download(report_file)

print("\n✓ Rapor indirme tamamlandı!")

## 🔧 Hızlı Test (Opsiyonel)

Tüm pipeline'ı çalıştırmadan önce hızlı test yapmak isterseniz:

In [ ]:
# HIZLI TEST: Sadece NMDA + tek yapı (7EU8)
# Düşük exhaustiveness ile (~10 dakika)

print("🧪 Hızlı test modu\n")

# 1. Tek yapı indir
!python scripts/1_download_structures.py

# 2. Ligand hazırla
!python scripts/2_prepare_ligand.py

# 3. Proteinleri hazırla
!python scripts/3_prepare_proteins.py

# 4. Config'i düzenle - düşük exhaustiveness
import yaml

with open('configs/nmda_targets.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Sadece 7EU8'i tut, exhaustiveness düşür
config['structures'] = {
    '7EU8': config['structures']['7EU8']
}
config['structures']['7EU8']['exhaustiveness'] = 8
config['structures']['7EU8']['num_modes'] = 5

with open('configs/nmda_targets.yaml', 'w') as f:
    yaml.dump(config, f)

print("\n✓ Test konfigürasyonu hazır")
print("Şimdi docking'i çalıştırın (scripts/4_run_docking.py)")

## 💡 İpuçları

1. **Runtime Süresi**: Colab ücretsiz max 12 saat. Pipeline 1-2 saatte biter.

2. **Sonuçları Hemen İndirin**: Colab geçici storage kullanır, session kapanınca silinir.

3. **Yeniden Başlatma**: Eğer bağlantı koptu:
   - Sonuçlar kaybolabilir
   - Eğer docking tamamlanmışsa, sadece analiz adımını çalıştırın

4. **Hız Artırma**:
   - `exhaustiveness` değerini düşürün (24 → 16)
   - Daha az yapı kullanın (her hedeften 1 tane)

5. **GPU**: Bu pipeline CPU tabanlı, GPU gerekmez.

## 📚 Dokümantasyon

- [README.md](https://github.com/egundeger/comp-chem-scripts/blob/main/ketamine-docking/README.md) - Kapsamlı dokümantasyon
- [QUICKSTART.md](https://github.com/egundeger/comp-chem-scripts/blob/main/ketamine-docking/QUICKSTART.md) - Hızlı başlangıç
- [README_COLAB.md](https://github.com/egundeger/comp-chem-scripts/blob/main/ketamine-docking/README_COLAB.md) - Colab kılavuzu

## ❓ Sorun Giderme

**"Vina not found" hatası:**
```python
!which vina
# Eğer bulunamazsa, Vina kurulum hücresini tekrar çalıştırın
```

**"Module not found" hatası:**
```python
!pip install <eksik_paket>
```

**"ketamine-docking directory not found":**
```python
# Branch ismini kontrol edin, doğru branch'i clone edin
!git clone -b claude/ketamine-docking-pipeline-011CV5ZKC6fFJBRhdCG2Mtt6 https://github.com/egundeger/comp-chem-scripts.git
```

**Docking çok uzun sürüyor:**
- Config dosyalarında `exhaustiveness` değerini düşürün
- Daha az yapı kullanın

**Runtime disconnect:**
- Sonuçları sık sık indirin
- Colab Pro kullanarak daha uzun runtime alabilirsiniz